# A/B Test — Enterprise Feature Analysis

Simulates and analyzes a two-sided A/B experiment modeled on an enterprise SaaS feature launch.
The dataset is synthetic with controlled ground truth — meaning the 'right answer' is known upfront,
which makes it possible to evaluate whether the statistical methods actually detect what they should.

The analysis runs in stages: power analysis → data generation → primary tests → robustness check → sequential monitoring → visualization.

## Stage 0: Power Analysis — How Many Users Do We Need?

Before running the experiment, we need to know how large a sample is required to reliably detect the effect we care about.
Too small and we'll miss real effects. Too large and we're wasting experiment runtime.

We're targeting an 8 pp lift (0.22 → 0.30 baseline), with 80% power and α = 0.05.
This is the step that justifies using n=1500 per group — it's not arbitrary.

In [ ]:
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

baseline = 0.22
target = 0.30
alpha = 0.05
power = 0.80

effect_size = proportion_effectsize(target, baseline)
analysis = NormalIndPower()
n_required = analysis.solve_power(effect_size=effect_size, alpha=alpha, power=power)

print(f"Baseline rate:         {baseline:.0%}")
print(f"Target rate:           {target:.0%}")
print(f"Cohen's h (effect):    {effect_size:.4f}")
print(f"Required per group:    {n_required:.0f}")
print(f"Total sample needed:   {n_required * 2:.0f}")
print(f"\nUsing n=1500/group gives power: {analysis.solve_power(effect_size=effect_size, alpha=alpha, nobs1=1500):.2%}")

## Stage 1: Data Generation

User-level experiment data is generated by `ExperimentData` in `data_generator.py`.
It models random group assignment, binary activation outcomes, and continuous session behavior.

By setting a known baseline rate and true lift at generation time, we create controlled ground truth —
which means we can verify downstream whether the statistical tests correctly detect the effect.

In [ ]:
import numpy as np
import pandas as pd
from data_generator import ExperimentData

gen = ExperimentData(seed=42)
df = gen.generate()

print(f"Total users: {len(df)}")
print(f"Control:     {(df['group']=='control').sum()}")
print(f"Treatment:   {(df['group']=='treatment').sum()}")
df.head()

In [ ]:
df.groupby("group")[["activated", "sessions"]].mean()

## Stage 2: Z-Test — Did Activation Rate Change?

A two-proportion z-test checks whether the difference in activation rates between control and treatment
is statistically real or just noise.

In [ ]:
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest

ctrl = df[df['group'] == 'control']
treat = df[df['group'] == 'treatment']

n_ctrl = len(ctrl)
n_treat = len(treat)
x_ctrl = ctrl['activated'].sum()
x_treat = treat['activated'].sum()

z_stat, p_value = proportions_ztest([x_treat, x_ctrl], [n_treat, n_ctrl])

print(f"Control rate:   {x_ctrl/n_ctrl:.4f}")
print(f"Treatment rate: {x_treat/n_treat:.4f}")
print(f"Z-statistic:    {z_stat:.4f}")
print(f"P-value:        {p_value:.4f}")
print(f"Significant:    {p_value < 0.05}")

## Stage 3: Welch's T-Test + Cohen's d — Session Engagement

Welch's t-test is used here instead of Student's t because it doesn't assume equal variance between groups —
which is safer when treatment might change not just the mean but the spread of session behavior.
Cohen's d gives a scale-free effect size so we can ask whether the difference is practically meaningful, not just significant.

In [ ]:
t_stat, p_value_t = stats.ttest_ind(treat['sessions'], ctrl['sessions'], equal_var=False)

pooled_std = np.sqrt((ctrl['sessions'].std()**2 + treat['sessions'].std()**2) / 2)
cohens_d = (treat['sessions'].mean() - ctrl['sessions'].mean()) / pooled_std

print(f"Control mean sessions:   {ctrl['sessions'].mean():.4f}")
print(f"Treatment mean sessions: {treat['sessions'].mean():.4f}")
print(f"Difference:              {treat['sessions'].mean() - ctrl['sessions'].mean():.4f}")
print(f"T-statistic:             {t_stat:.4f}")
print(f"P-value:                 {p_value_t:.4f}")
print(f"Cohen's d:               {cohens_d:.4f}")
print(f"Significant:             {p_value_t < 0.05}")

## Stage 4: Chi-Square Test — Robustness Check

The chi-square test covers the same hypothesis as the z-test — whether activation rate differs by group.
Running both is intentional: in practice, teams often reach for chi-square out of habit when z-test is
more appropriate for proportions. Having both confirms they agree, and Cramér's V gives a scale-free
effect size that's easier to communicate to non-technical stakeholders than Cohen's h.

Bonferroni correction is applied here since we're testing two metrics (activation + sessions).
Adjusted α = 0.05 / 2 = 0.025.

In [ ]:
from scipy.stats import chi2_contingency

contingency = pd.crosstab(df['group'], df['activated'])
print("Contingency table:")
print(contingency)

chi2, p_chi, dof, expected = chi2_contingency(contingency)

n = len(df)
cramers_v = np.sqrt(chi2 / (n * (min(contingency.shape) - 1)))

alpha_bonferroni = 0.05 / 2

print(f"\nChi-square statistic: {chi2:.4f}")
print(f"P-value:              {p_chi:.4f}")
print(f"Cramér's V:           {cramers_v:.4f}")
print(f"Bonferroni α:         {alpha_bonferroni}")
print(f"Significant:          {p_chi < alpha_bonferroni}")

## Stage 5: O'Brien-Fleming Sequential Monitoring

In a live experiment, there's always pressure to check results early and stop if things look good.
But peeking repeatedly inflates false positives — if you stop the moment p < 0.05, you'll get fooled by noise.

O'Brien-Fleming boundaries solve this by requiring much stronger evidence early in the experiment,
then relaxing the threshold as more data accumulates. The boundary at each look scales as:
`z_crit * sqrt(n_total / n_current)` — so early looks need much higher z-stats to cross.

The experiment crosses the boundary at look 3 (30% enrolled). Worth noting: looks 1 and 2 stay
below the boundary despite a real effect being present — exactly the behavior OBF is designed to produce.

In [ ]:
from scipy.stats import norm

def obf_sequential(df, n_looks=10, alpha=0.05):
    n_total = len(df)
    z_crit = norm.ppf(1 - alpha/2)
    results = []

    for look in range(1, n_looks + 1):
        fraction = look / n_looks
        n_current = int(n_total * fraction)
        df_current = df.iloc[:n_current]

        ctrl_c = df_current[df_current['group'] == 'control']
        treat_c = df_current[df_current['group'] == 'treatment']

        x_c = ctrl_c['activated'].sum()
        x_t = treat_c['activated'].sum()
        n_c = len(ctrl_c)
        n_t = len(treat_c)

        p_c = x_c / n_c if n_c > 0 else 0
        p_t = x_t / n_t if n_t > 0 else 0
        p_pool = (x_c + x_t) / (n_c + n_t) if (n_c + n_t) > 0 else 0

        se = np.sqrt(p_pool * (1 - p_pool) * (1/n_c + 1/n_t)) if p_pool > 0 else 1
        z = (p_t - p_c) / se if se > 0 else 0

        boundary = z_crit * np.sqrt(n_total / n_current)
        crossed = abs(z) > boundary

        results.append({
            'look': look,
            'pct_enrolled': f"{int(fraction*100)}%",
            'z_stat': round(z, 3),
            'boundary': round(boundary, 3),
            'crossed': crossed
        })

    return pd.DataFrame(results)

seq_results = obf_sequential(df)
print(seq_results.to_string(index=False))

## Stage 6: Ship / Iterate / Kill Decision

Statistical significance alone isn't a ship decision. The framework here combines significance + effect size:

- **Ship**: significant AND effect size meets minimum bar (Cohen's d ≥ 0.2 for sessions, lift ≥ 5 pp for activation)
- **Iterate**: significant but effect size below bar — real effect, but too small to justify shipping as-is
- **Kill**: not significant — insufficient evidence of impact

In [ ]:
activation_lift_pp = (x_treat/n_treat) - (x_ctrl/n_ctrl)
significant_activation = p_value < 0.025  # Bonferroni adjusted
significant_sessions = p_value_t < 0.025

min_lift_pp = 0.05
min_cohens_d = 0.2

def decision(significant, effect_meets_bar):
    if not significant:
        return 'KILL'
    elif significant and effect_meets_bar:
        return 'SHIP'
    else:
        return 'ITERATE'

act_decision = decision(significant_activation, activation_lift_pp >= min_lift_pp)
sess_decision = decision(significant_sessions, cohens_d >= min_cohens_d)

print(f"Activation lift:  {activation_lift_pp:.3f} ({activation_lift_pp*100:.1f} pp) → {act_decision}")
print(f"Session Cohen's d: {cohens_d:.3f}               → {sess_decision}")
print(f"\nOverall recommendation: {'SHIP' if act_decision == 'SHIP' and sess_decision == 'SHIP' else 'ITERATE'}")

## Stage 7: Visualizations

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

fig = plt.figure(figsize=(14, 10))
gs = gridspec.GridSpec(2, 2, hspace=0.4, wspace=0.35)

# Plot 1: Activation rates
ax1 = fig.add_subplot(gs[0, 0])
groups_labels = ['Control', 'Treatment']
rates = [ctrl['activated'].mean(), treat['activated'].mean()]
colors = ['#5B8DB8', '#2E7D32']
bars = ax1.bar(groups_labels, rates, color=colors, width=0.5, edgecolor='white')
for bar, rate in zip(bars, rates):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
             f'{rate:.1%}', ha='center', va='bottom', fontweight='bold')
ax1.set_ylim(0, 0.4)
ax1.set_title('Activation Rate by Group', fontweight='bold')
ax1.set_ylabel('Activation Rate')
ax1.spines[['top', 'right']].set_visible(False)

# Plot 2: Session distribution
ax2 = fig.add_subplot(gs[0, 1])
ax2.hist(ctrl['sessions'], bins=30, alpha=0.6, color='#5B8DB8', label='Control', density=True)
ax2.hist(treat['sessions'], bins=30, alpha=0.6, color='#2E7D32', label='Treatment', density=True)
ax2.axvline(ctrl['sessions'].mean(), color='#1565C0', linestyle='--', linewidth=1.5)
ax2.axvline(treat['sessions'].mean(), color='#1B5E20', linestyle='--', linewidth=1.5)
ax2.set_title('Session Distribution', fontweight='bold')
ax2.set_xlabel('Sessions per User')
ax2.set_ylabel('Density')
ax2.legend(fontsize=9)
ax2.spines[['top', 'right']].set_visible(False)

# Plot 3: Effect sizes summary
ax3 = fig.add_subplot(gs[1, 0])
metrics = ["Activation\n(Cohen's h)", "Sessions\n(Cohen's d)"]
from statsmodels.stats.proportion import proportion_effectsize
cohens_h = proportion_effectsize(treat['activated'].mean(), ctrl['activated'].mean())
effect_sizes = [abs(cohens_h), cohens_d]
bar_colors = ['#E65100' if e >= 0.2 else '#B0BEC5' for e in effect_sizes]
bars3 = ax3.barh(metrics, effect_sizes, color=bar_colors, edgecolor='white')
ax3.axvline(0.2, color='gray', linestyle='--', linewidth=1, label='Small effect threshold (0.2)')
ax3.axvline(0.5, color='gray', linestyle=':', linewidth=1, label='Medium effect threshold (0.5)')
for bar, val in zip(bars3, effect_sizes):
    ax3.text(val + 0.01, bar.get_y() + bar.get_height()/2, f'{val:.3f}', va='center', fontweight='bold')
ax3.set_title('Effect Sizes', fontweight='bold')
ax3.set_xlabel('Effect Size')
ax3.legend(fontsize=8)
ax3.spines[['top', 'right']].set_visible(False)

# Plot 4: OBF sequential monitoring
ax4 = fig.add_subplot(gs[1, 1])
looks = seq_results['look'].tolist()
z_stats = seq_results['z_stat'].tolist()
boundaries = seq_results['boundary'].tolist()
ax4.plot(looks, z_stats, 'o-', color='#2E7D32', linewidth=2, markersize=6, label='Z-statistic')
ax4.plot(looks, boundaries, '--', color='#C62828', linewidth=1.5, label='OBF boundary')
ax4.plot(looks, [-b for b in boundaries], '--', color='#C62828', linewidth=1.5)
ax4.fill_between(looks, boundaries, [max(z_stats)+1]*len(looks), alpha=0.08, color='#C62828')
ax4.fill_between(looks, [-b for b in boundaries], [-(max(z_stats)+1)]*len(looks), alpha=0.08, color='#C62828')
ax4.set_title('Sequential Monitoring (OBF)', fontweight='bold')
ax4.set_xlabel('Interim Look')
ax4.set_ylabel('Z-statistic')
ax4.legend(fontsize=8)
ax4.spines[['top', 'right']].set_visible(False)

plt.suptitle('A/B Test Results — Enterprise Feature Launch',
             fontsize=14, fontweight='bold', y=1.01)

plt.savefig('ab_test_results.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved to ab_test_results.png")